In [ ]:
import copy

import torch
import torch.nn as nn
import torchvision
import numpy as np
from matplotlib import pyplot as plt

from search.parallel_gradient import ParallelGradientDescent
from utils.sampling import BatchNegativeSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "modelnet10"
architecture = "pointnetplus_pca_then_norm_randomize"
budget = 120

In [ ]:
import model.pointnet_plus

In [ ]:
from model.pointnet_plus import PointNetPlus

In [ ]:
from experiment_thesis.dataset_preperation.get_dataset import get_dataset_info,get_dataset

dataset_info = get_dataset_info(dataset)
dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data,batch_size=32)

In [ ]:
dataset_info

dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
batch_size = next(iter(test_loader))[0].shape[0]


In [ ]:
batch_size

In [ ]:
batch_size = next(iter(train_loader))[0].shape[0]


In [ ]:
from utils.eval.vis import vis_dataset

vis_dataset(train_loader,val_loader,test_loader_transformed)

In [ ]:
from experiment_thesis.main import train_and_get_model,train_or_load_energy_model
from experiment_thesis.dataset_preperation.basic_networks import get_network
from utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "comparision_over_budget")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, f"{safe}.json")

In [ ]:
from dataset.geometric_wrapper import BatchNormalizeScale, NormalizeRotationVectorizedModule
from dataset.geometric_wrapper import TensorGeometricModelWrapper


core = PointNetPlus(num_classes=dataset_info.num_classes)
pre = []

pre.append(BatchNormalizeScale())

core = nn.Sequential(*pre, core)



model = TensorGeometricModelWrapper(core)

modelname = f"{dataset}_pointnet_plus_norm"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "32",
},load_if_exists=True)
model.cuda().eval()

res = evaluate_base_model(model, test_loader, device)
print(res)
print()
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)


In [ ]:
from dataset.geometric_wrapper import BatchNormalizeScale, NormalizeRotationVectorizedModule
from dataset.geometric_wrapper import TensorGeometricModelWrapper


core = PointNetPlus(num_classes=dataset_info.num_classes)
pre = []

pre.append(BatchNormalizeScale())
pre.append(
NormalizeRotationVectorizedModule(
            max_points=-1,
            sort=False,
            ensure_proper_rotation=False,
            randomize=False,)
)
core = nn.Sequential(*pre, core)



model = TensorGeometricModelWrapper(core)

modelname = f"{dataset}_pointnet_plus_norm_then_pca"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "32",
},load_if_exists=True)
model.cuda().eval()

res = evaluate_base_model(model, test_loader, device)
print(res)
print()
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)


In [ ]:
from dataset.geometric_wrapper import BatchNormalizeScale, NormalizeRotationVectorizedModule
from dataset.geometric_wrapper import TensorGeometricModelWrapper


core = PointNetPlus(num_classes=dataset_info.num_classes)
pre = []
pre.append(
NormalizeRotationVectorizedModule(
            max_points=-1,
            sort=False,
            ensure_proper_rotation=False,
            randomize=False,)
)

pre.append(BatchNormalizeScale())

core = nn.Sequential(*pre, core)



model = TensorGeometricModelWrapper(core)

modelname = f"{dataset}_pointnet_plus_pca_then_norm"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "32",
},load_if_exists=True)
model.cuda().eval()

res = evaluate_base_model(model, test_loader, device)
print(res)
print()
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)


In [ ]:
from dataset.geometric_wrapper import BatchNormalizeScale, NormalizeRotationVectorizedModule
from dataset.geometric_wrapper import TensorGeometricModelWrapper


core = PointNetPlus(num_classes=dataset_info.num_classes)
pre = []

pre.append(BatchNormalizeScale())
pre.append(
NormalizeRotationVectorizedModule(
            max_points=-1,
            sort=False,
            ensure_proper_rotation=False,
            randomize=True,)
)
core = nn.Sequential(*pre, core)



model = TensorGeometricModelWrapper(core)

modelname = f"{dataset}_pointnet_plus_norm_then_pca_random"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "32",
},load_if_exists=True)
model.cuda().eval()

res = evaluate_base_model(model, test_loader, device)
print(res)
print()
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)


In [ ]:
from dataset.geometric_wrapper import BatchNormalizeScale, NormalizeRotationVectorizedModule
from dataset.geometric_wrapper import TensorGeometricModelWrapper


core = PointNetPlus(num_classes=dataset_info.num_classes)
pre = []
pre.append(
NormalizeRotationVectorizedModule(
            max_points=-1,
            sort=False,
            ensure_proper_rotation=False,
            randomize=True,)
)

pre.append(BatchNormalizeScale())

core = nn.Sequential(*pre, core)



model = TensorGeometricModelWrapper(core)

modelname = f"{dataset}_pointnet_plus_pca_then_norm_randomize"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "32",
},load_if_exists=True)
model.cuda().eval()

res = evaluate_base_model(model, test_loader, device)
print(res)
print()
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)


In [ ]:


from dataset.geometric_wrapper import BatchNormalizeScale, NormalizeRotationVectorizedModule
from dataset.geometric_wrapper import TensorGeometricModelWrapper


core = PointNetPlus(num_classes=dataset_info.num_classes)
pre = []

pre.append(BatchNormalizeScale())
pre.append(
NormalizeRotationVectorizedModule(
            max_points=-1,
            sort=True,
            ensure_proper_rotation=True,
            randomize=True,)
)
core = nn.Sequential(*pre, core)



model = TensorGeometricModelWrapper(core)

modelname = f"{dataset}_pointnet_plus_norm_then_pca_random_reduced"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "32",
},load_if_exists=True)
model.cuda().eval()

res = evaluate_base_model(model, test_loader, device)
print(res)
print()
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)



In [ ]:


from dataset.geometric_wrapper import BatchNormalizeScale, NormalizeRotationVectorizedModule
from dataset.geometric_wrapper import TensorGeometricModelWrapper


core = PointNetPlus(num_classes=dataset_info.num_classes)
pre = []
pre.append(
NormalizeRotationVectorizedModule(
            max_points=-1,
            sort=True,
            ensure_proper_rotation=True,
            randomize=True,)
)
pre.append(BatchNormalizeScale())

core = nn.Sequential(*pre, core)



model = TensorGeometricModelWrapper(core)

modelname = f"{dataset}_pointnet_plus_pca_then_norm_random_reduced"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "32",
},load_if_exists=True)
model.cuda().eval()
res = evaluate_base_model(model, test_loader, device)
print(res)
print()
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)



In [ ]:


from dataset.geometric_wrapper import BatchNormalizeScale, NormalizeRotationVectorizedModule
from dataset.geometric_wrapper import TensorGeometricModelWrapper


core = PointNetPlus(num_classes=dataset_info.num_classes)
pre = []

pre.append(BatchNormalizeScale())
pre.append(
NormalizeRotationVectorizedModule(
            max_points=-1,
            sort=True,
            ensure_proper_rotation=True,
            randomize=False,)
)
core = nn.Sequential(*pre, core)



model = TensorGeometricModelWrapper(core)

modelname = f"{dataset}_pointnet_plus_norm_then_pca_reduced"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "32",
},load_if_exists=True)
model.cuda().eval()

res = evaluate_base_model(model, test_loader, device)
print(res)
print()
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)



In [ ]:


from dataset.geometric_wrapper import BatchNormalizeScale, NormalizeRotationVectorizedModule
from dataset.geometric_wrapper import TensorGeometricModelWrapper


core = PointNetPlus(num_classes=dataset_info.num_classes)
pre = []
pre.append(
NormalizeRotationVectorizedModule(
            max_points=-1,
            sort=True,
            ensure_proper_rotation=True,
            randomize=False,)
)
pre.append(BatchNormalizeScale())

core = nn.Sequential(*pre, core)



model = TensorGeometricModelWrapper(core)

modelname = f"{dataset}_pointnet_plus_pca_then_norm_reduced"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "32",
},load_if_exists=True)
model.cuda().eval()
res = evaluate_base_model(model, test_loader, device)
print(res)
print()
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)



In [ ]:


from dataset.geometric_wrapper import BatchNormalizeScale, NormalizeRotationVectorizedModule
from dataset.geometric_wrapper import TensorGeometricModelWrapper


core = PointNetPlus(num_classes=dataset_info.num_classes)
pre = []

pre.append(BatchNormalizeScale())
pre.append(
NormalizeRotationVectorizedModule(
            max_points=-1,
            sort=True,
            ensure_proper_rotation=True,
            randomize=False,fix_sign=True)
)
core = nn.Sequential(*pre, core)



model = TensorGeometricModelWrapper(core)

modelname = f"{dataset}_pointnet_plus_norm_then_pca_fixed_augmented"

train_and_get_model(model,model_dir_path,modelname, train_loader_transformed, val_loader_transformed , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "32",
},load_if_exists=True)
model.cuda().eval()

res = evaluate_base_model(model, test_loader, device)
print(res)
print()
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)



In [ ]:


from dataset.geometric_wrapper import BatchNormalizeScale, NormalizeRotationVectorizedModule
from dataset.geometric_wrapper import TensorGeometricModelWrapper


core = PointNetPlus(num_classes=dataset_info.num_classes)
pre = []

pre.append(BatchNormalizeScale())
pre.append(
NormalizeRotationVectorizedModule(
            max_points=-1,
            sort=True,
            ensure_proper_rotation=True,
            randomize=False,)
)
core = nn.Sequential(*pre, core)



model = TensorGeometricModelWrapper(core)

modelname = f"{dataset}_pointnet_plus_norm_then_pca_reduced_augmented"

train_and_get_model(model,model_dir_path,modelname, train_loader_transformed, val_loader_transformed , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "32",
},load_if_exists=True)
model.cuda().eval()

res = evaluate_base_model(model, test_loader, device)
print(res)
print()
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)



In [ ]:
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)

In [ ]:
res = evaluate_base_model(model, test_loader, device)
print(res)

In [ ]:
res = evaluate_base_model(model, train_loader_transformed, device)
print(res)

In [ ]:
class TensorGeometricModelUnwrapper(torch.nn.Module):
    """
    Wrapper for a torch_geometric model that receives a tuple of (pos, y) as input and creates
    a Data object from it that is passed to the model.
    """
    def __init__(self):
        super(TensorGeometricModelUnwrapper, self).__init__()

    def forward(self, data):
        # pos and y are batched tensors from a DataLoader
        # need to reconstruct the original Data objects for torch_geometric models

        pos = data.pos
        batch = data.batch
        #split pos into individual tensors based on batch
        pos_list = torch.split(pos, torch.bincount(batch).tolist())
        return torch.stack(pos_list)

In [ ]:

post_transform = TensorGeometricModelWrapper(model.model[:-1])
post_transform = torch.nn.Sequential(
    post_transform,
    TensorGeometricModelUnwrapper()
)

In [ ]:
post_transform[0].train()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Get one batch from the loader
data = next(iter(test_loader))
x = data[0].to(device)

# Number of times to apply post_transform
num_plots = 3
idx = 1  # sample index from batch

# Create subplot figure
fig = make_subplots(
    rows=1, cols=num_plots,
    specs=[[{'type':'scatter3d'}]*num_plots],
    subplot_titles=[f"Transformed {i+1}" for i in range(num_plots)]
)

for i in range(num_plots):
    # Apply random transform each time
    x_transformed = post_transform(x).cpu().detach().numpy()

    fig.add_trace(
        go.Scatter3d(
            x=x_transformed[idx, :, 0],
            y=x_transformed[idx, :, 1],
            z=x_transformed[idx, :, 2],
            mode='markers',
            marker=dict(
                size=2,
                color=x_transformed[idx, :, 2],  # color by z-axis
                colorscale='Viridis',
                opacity=0.8
            )
        ),
        row=1, col=i+1
    )

fig.update_layout(margin=dict(l=0, r=0, b=0, t=30), height=500, width=1500)
fig.show()


In [ ]:
dsfhgdh

In [ ]:
from experiment_thesis.dataset_preperation.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method
    ).cuda()

In [ ]:
from experiment_thesis.dataset_preperation.basic_networks import get_network_layer

layer,layer_io = get_network_layer(dataset_info, architecture, 1, num_classes=None, num_rotations=8)

In [ ]:
from confidence.direct.logit_based import EnergyConfidence
from utils.transformation_problem import TransformationProblem

problem_energy = TransformationProblem(EnergyConfidence(), transform_seq,
                                                      consolidate_method="consolidate_simple")

In [ ]:
#create a second classifier model with just a single output
energy_model = get_network(dataset_info,architecture, num_classes=1).to(device)

In [ ]:
def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


from utils.augments import build_default_augmentations, small_affine_augment_2d
from utils.sampling_strategy import GaussianSamplingStrategyLatent, TransformLatentSamplingStrategy
import importlib
import utils.sampling_strategy
import utils.sampling

importlib.reload(utils.sampling)
from utils.sampling import BatchNegativeSampler



In [ ]:
#test ot
from search.shgo import SHGO
random_search = SHGO(initial_samples=120, local_max_steps=0)

In [ ]:
from utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search,evaluate_confidence_and_search


In [ ]:
model.eval().cuda()

In [ ]:
#chek if data is iamge data
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]

In [ ]:
is_image_data

In [ ]:
#unload main model from gpu
model.cuda()

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#build smaller train
small_train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size//2, shuffle=True, num_workers=4,persistent_workers=True)

In [ ]:
small_val_loader = torch.utils.data.DataLoader(dataset_val, batch_size=batch_size//2, shuffle=False, num_workers=4,persistent_workers=True)

In [ ]:
from utils.augments import ComposeAugmentations, random_gaussian_noise, random_contrast, \
    random_gamma  ,random_blur_or_sharpen,build_default_augmentations
import utils.augments


energy_model2 = get_network(dataset_info,architecture, num_classes=1).to(device)

from experiment_thesis.main import train_or_load_energy_model

if is_image_data:
    transform_true_function = small_affine_augment_2d
    affine_augment = utils.augments.build_default_augmentations()
else:
    transform_true_function = None
    affine_augment = None

negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    =transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)

energy_conf2 = train_or_load_energy_model(
    energy_model2, model_dir_path, f"{modelname}_energy2", small_train_loader,
    small_val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": 50,
        "precision": "16-mixed",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)


In [ ]:
test_loader_transformed_smaller = torch.utils.data.DataLoader(dataset_test, batch_size=batch_size//32, shuffle=True, num_workers=4,persistent_workers=True)

In [ ]:
random_search = SHGO(initial_samples=240, local_max_steps=0)

In [ ]:
model.cuda().eval()

In [ ]:
from model.pointnet_plus import SAModule
def set_deterministic_fps(model, random_start=False):
    for module in model.modules():
        if isinstance(module, SAModule):
            module.random_start = random_start
            print(f"Set random_start={random_start} for {module.__class__.__name__}")

set_deterministic_fps(model)

In [ ]:
energy_conf2.cuda().eval()
problem_energy2 = TransformationProblem(energy_conf2, transform_seq,                                              consolidate_method="consolidate_simple")



In [ ]:
evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_energy2,
    test_loader=test_loader_transformed_smaller, max_batch_override=32,
    repeats=1)

In [ ]:
#disable sa samling


In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from torch.utils.data import SequentialSampler
from embedding_cache import LayerEmbeddingCache

transform_name = dataset_info.transform_seq_name

cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"

cache_train = LayerEmbeddingCache(model, train_loader_no_shuffle,
                                  cache_dir=os.path.join(embedding_cache_path, cache_name_train))

dual_output_model = cache_train.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)
embeddings_t, final_t, classes_t = cache_train.__call__(layer, capture_modes=layer_io, flatten=True)



from utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import SplitConfidence, PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import KNNConfidence, PerClassKNNConfidence

from confidence.input_transform import InputTransformImage, PCAInputModule, RandomProjectionModule

input_transform_image = InputTransformImage((3, 3), (128, 7, 7))
input_transform_pca = PCAInputModule(512)
input_transform_random = RandomProjectionModule(512, method="gaussian")

nn_pytorch_pretrained = KNNConfidence(metric="cosine", input_transform=None)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.cuda()

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.0)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple")


In [ ]:
model.eval().cuda()

In [ ]:
train_loader_transformed_smaller = torch.utils.data.DataLoader(dataset_train, batch_size=batch_size//32, shuffle=True, num_workers=4,persistent_workers=True)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
random_search_huge = SHGO(initial_samples=240,local_runs=1, local_max_steps=0,include_zero_always=False)

In [ ]:
evaluate_confidence_and_search(
    model, optimizer=random_search_huge, problem=problem_nn_pytorch_pretrained,
    test_loader=train_loader_no_shuffle, max_batch_override=32,
    repeats=1)

In [ ]:
#test wether model is deterministic

In [ ]:
x = next(iter(train_loader_no_shuffle))

In [ ]:
dual_output_model(x[0].to(device))[0].cpu().detach().numpy()

In [ ]:
embeddings_t[0].cpu().detach().numpy()

In [ ]:
logit_energy = SinglePassConfidence(model, EnergyConfidence(), index=None)
problem_energy_logits = TransformationProblem(logit_energy, transform_seq,                                              consolidate_method="consolidate_simple")

In [ ]:
evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_energy_logits,
    test_loader=test_loader_transformed_smaller, max_batch_override=32,
    repeats=1)